# A/B Testing
> Compare two pipeline configurations with statistical rigour.

`ABTestRunner` evaluates two systems on the same samples and
determines a winner using aggregate metrics and a p-value
for statistical significance.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.evaluation import (
    ABTestRunner,
    PipelineEvaluator,
    RetrievalMetricsCalculator,
    LLMRAGEvaluator,
)
from anchor.evaluation.models import (
    ABTestResult,
    EvaluationSample,
    RAGMetrics,
)
from anchor.models import ContextItem, SourceType

## Define Two Evaluation Functions
System A returns higher scores than System B, simulating
a better-performing pipeline.

In [ ]:
def system_a_judge(query, answer, contexts, ground_truth=None):
    return RAGMetrics(
        faithfulness=0.92,
        relevancy=0.88,
        context_precision=0.85,
        context_recall=0.80,
    )

def system_b_judge(query, answer, contexts, ground_truth=None):
    return RAGMetrics(
        faithfulness=0.70,
        relevancy=0.65,
        context_precision=0.60,
        context_recall=0.55,
    )

print("System A: high-quality mock scores")
print("System B: lower-quality mock scores")

## Build Two Pipeline Evaluators

In [ ]:
calculator = RetrievalMetricsCalculator()

pipeline_a = PipelineEvaluator(
    retrieval_calculator=calculator,
    rag_evaluator=LLMRAGEvaluator(eval_fn=system_a_judge),
)
pipeline_b = PipelineEvaluator(
    retrieval_calculator=calculator,
    rag_evaluator=LLMRAGEvaluator(eval_fn=system_b_judge),
)

print(f"Pipeline A ready: {type(pipeline_a).__name__}")
print(f"Pipeline B ready: {type(pipeline_b).__name__}")

## Prepare Test Samples
Both systems will be evaluated on the same set of samples.

In [ ]:
def make_retrieved(n=5):
    return [
        ContextItem(
            id=f"doc-{i}",
            content=f"Content {i}",
            source=SourceType.RETRIEVAL,
            score=1.0 - i * 0.1,
            priority=5,
            token_count=10,
        )
        for i in range(n)
    ]

samples = [
    EvaluationSample(
        query=f"Question {i}?",
        answer=f"Answer {i}.",
        contexts=[f"Context for question {i}."],
        retrieved=make_retrieved(),
        relevant=["doc-0", "doc-1"],
        ground_truth=f"Ground truth {i}.",
    )
    for i in range(10)
]

print(f"Test samples: {len(samples)}")

## Create and Run the A/B Test
`ABTestRunner` takes a base evaluator and the number of samples.

In [ ]:
runner = ABTestRunner(
    evaluator=pipeline_a,
    num_samples=len(samples),
)

print(f"ABTestRunner created")
print(f"  Evaluator type: {type(runner.evaluator).__name__}")
print(f"  Num samples:    {runner.num_samples}")

## Inspect ABTestResult Schema
The result exposes per-system metrics, the winner, and a
p-value for significance.

In [ ]:
# In a full run you would call:
#   result = runner.run(
#       system_a=pipeline_a,
#       system_b=pipeline_b,
#       samples=samples,
#   )
#
# For demonstration, we show the expected result structure:

print("ABTestResult fields:")
print("  .system_a_metrics  -- aggregated metrics for system A")
print("  .system_b_metrics  -- aggregated metrics for system B")
print("  .winner            -- 'A', 'B', or 'tie'")
print("  .p_value           -- statistical significance")

In [ ]:
# Simulated result for display purposes
print("\nSimulated A/B test result:")
print(f"  System A faithfulness: 0.92")
print(f"  System B faithfulness: 0.70")
print(f"  Winner:  A")
print(f"  p-value: 0.003")

## Key Takeaways

- `ABTestRunner` compares two pipeline configurations on identical data.
- The result includes per-system aggregate metrics, a declared
  winner, and a p-value for confidence.
- Run A/B tests before deploying pipeline changes to production.
- Increase `num_samples` for tighter statistical confidence.